# Stage 1 — Coronary segmentation: Colab BUILD side

**Split:** this notebook runs on **Colab (GPU)** and produces the portable handoff: a distilled student `state_dict` (`student.pt`) **+** `student.onnx`. CoreML conversion, weight compression, the clDice gate and on-device benchmark happen **on the Mac** (see `docs/COLAB_MAC_SPLIT.md`) — coremltools can convert on Linux but you can only *run/benchmark* CoreML on macOS.

**Flow:** mount Drive → install → nnU-Net teacher → cache teacher logits → distill TinyU-Net student → eval (Dice/clDice/HD95) → export ONNX + state_dict to Drive.

**Two Colab gotchas handled below:** (1) ephemeral FS → everything heavy points at Drive; (2) session limits → nnU-Net epochs cut + checkpoint-resume.

In [ ]:
# 1) Mount Drive (persistent storage; Colab wipes local FS on disconnect)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/intv-img'   # <- your project root on Drive
os.makedirs(DRIVE, exist_ok=True)
print('project root:', DRIVE)

In [ ]:
# 2) Get the repo + install the BUILD env (torch-CUDA + nnunetv2 + ultralytics)
#    Option A: clone your repo;  Option B: keep the repo on Drive and symlink.
REPO = f'{DRIVE}/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    # TODO: replace with your repo URL, or upload the folder to Drive
    !git clone <YOUR_REPO_URL> {REPO}
%cd {REPO}
!pip -q install nnunetv2>=2.5 monai>=1.3 ultralytics>=8.2 SimpleITK onnx onnxruntime \
    opencv-python scikit-image scikit-learn pyyaml tqdm
print('build env ready (CoreML/coremltools NOT needed here — that is the Mac side)')

In [ ]:
# 3) Point nnU-Net + run artifacts at DRIVE so a dropped session does not lose the teacher
for k, sub in [('nnUNet_raw','nnUNet_raw'), ('nnUNet_preprocessed','nnUNet_preprocessed'),
               ('nnUNet_results','nnUNet_results')]:
    p = f'{DRIVE}/{sub}'; os.makedirs(p, exist_ok=True); os.environ[k] = p
RUNS = f'{DRIVE}/runs/coronary'; os.makedirs(RUNS, exist_ok=True)
TEACHER_CACHE = f'{DRIVE}/teacher_cache/coronary'; os.makedirs(TEACHER_CACHE, exist_ok=True)
print('nnUNet_raw       =', os.environ['nnUNet_raw'])
print('runs (student)   =', RUNS)
print('teacher logits   =', TEACHER_CACHE)

## Data — place under Drive, then standardize
Open datasets (no gating): **ARCADE** (Zenodo 8386059/10390295) and **DCA1** (CIMAT). Put raw under `data/raw/` (here repointed to Drive). The `make prep-coronary` step runs the two converters (`arcade_to_coco.py`, `dca1_to_nnunet.py`) — fill their TODOs with the dataset-specific parsing.

In [ ]:
# 4) Download / place raw data on Drive (idempotent). Replace with real fetch.
RAW = f'{DRIVE}/data/raw'; os.makedirs(RAW, exist_ok=True)
# TODO ARCADE: download Zenodo record 8386059 -> {RAW}/arcade  (task1 seg + task2 stenosis)
# TODO DCA1:   download CIMAT DB_Angiograms -> {RAW}/dca1     (134 imgs, binary vessel masks)
!ls -la {RAW} || true
# symlink Drive raw into the repo's expected data/raw so configs resolve unchanged
!mkdir -p data && ln -sfn {RAW} data/raw && ls -la data/raw

In [ ]:
# 5) Standardize -> COCO + nnU-Net splits with CLAHE (fills data/processed)
!python -m src.data_prep.arcade_to_coco --config configs/coronary_seg.yaml
!python -m src.data_prep.dca1_to_nnunet --config configs/coronary_seg.yaml
# Smoke-test the runnable metrics now (Stage 0 exit)
!python -m src.eval.metrics

## Teacher — nnU-Net v2 (ResEncM, 2D)
Sets the accuracy ceiling + doubles as a label-quality oracle. Free-tier **T4 (16 GB)** clears the 2D ResEncM floor; Pro (L4/A100) is just faster. nnU-Net default 1000 epochs is long — cut for a v1 result and rely on checkpoint-resume across sessions (state lives on Drive).

In [ ]:
# 6) Teacher train (subprocess CLI). DATASET_ID set during prep (nnU-Net Dataset0XX_Coronary).
DATASET_ID = '001'   # <- match what dca1_to_nnunet/arcade_to_coco registered
EPOCHS = 250         # cut from 1000 for v1; raise once the path is proven
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity
# --c resumes from the last checkpoint on Drive if the session dropped
!nnUNetv2_train {DATASET_ID} 2d 0 -tr nnUNetTrainer --c || nnUNetv2_train {DATASET_ID} 2d 0 -tr nnUNetTrainer

In [ ]:
# 7) Cache teacher LOGITS on the training images (the distillation target).
#    nnUNetv2_predict --save_probabilities writes per-case softmax/logits .npz to Drive.
!nnUNetv2_predict -i $nnUNet_raw/Dataset{DATASET_ID}_Coronary/imagesTr \
                  -o {TEACHER_CACHE} -d {DATASET_ID} -c 2d -f 0 --save_probabilities
!ls {TEACHER_CACHE} | head

## Distill — TinyU-Net student (~2-3M params, edge)
`src/models/distill.py` provides `kd_loss` (binary sigmoid-KD) + `distill()`. The dataloader yields `(image, mask, teacher_logits)`; teacher logits come from the cache above. Student `state_dict` is the portable Mac handoff.

In [ ]:
# 8) Build the (image, mask, teacher_logit) loader, then distill.
import torch, yaml
from torch.utils.data import DataLoader
from src.models.seg_student import TinyUNet
from src.models.distill import distill, TeacherCacheDataset
from src.eval.metrics import dice, cldice

cfg = yaml.safe_load(open('configs/coronary_seg.yaml'))
ds = TeacherCacheDataset(processed_dir='data/processed/coronary',
                         teacher_cache=TEACHER_CACHE, size=cfg['preprocess']['size'])
loader = DataLoader(ds, batch_size=cfg['train']['batch'], shuffle=True, num_workers=2)

student = TinyUNet(in_ch=1, n_classes=1,
                   base=cfg['student']['base_ch'], depth=cfg['student'].get('depth', 4))

def quick_eval(m):                       # called every 10 epochs
    m.eval()
    with torch.no_grad():
        x, y, _ = ds[0]
        p = (torch.sigmoid(m(x[None].cuda())).cpu().numpy().squeeze() >= 0.5)
    m.train()
    g = y.numpy().squeeze()
    return f'Dice {dice(p, g):.3f} clDice {cldice(p, g):.3f}'

distill(student, loader,
        epochs=cfg['train']['epochs'], lr=cfg['train']['lr'],
        alpha=cfg['distill']['alpha'], T=cfg['distill']['temperature'],
        eval_fn=quick_eval, ckpt=f'{RUNS}/student.pt')

## Export handoff — ONNX + state_dict to Drive
ONNX is for non-Apple targets / onnxruntime checks. The **state_dict** (`student.pt`) is what the Mac rebuilds into TinyU-Net before CoreML conversion.

In [ ]:
# 9) Export ONNX (state_dict already saved on Drive by distill())
!python -m src.export.to_onnx --weights {RUNS}/student.pt --out {RUNS}/student.onnx
# Optional: ONNX INT8 sanity on Colab (non-Apple path)
!python -m src.export.quantize_int8 --onnx {RUNS}/student.onnx
!python -m src.eval.edge_benchmark --model {RUNS}/student.onnx
print('\nHANDOFF READY on Drive:')
print(' ', f'{RUNS}/student.pt    (state_dict -> Mac CoreML)')
print(' ', f'{RUNS}/student.onnx  (non-Apple / onnxruntime)')

## Next — Mac side (Apple silicon)
Pull `student.pt` from Drive, then on the Mac:
```bash
make export-coreml   MODEL=runs/coronary/student.pt          # -> student.mlpackage (palettized)
make validate-coreml CORE=runs/coronary/student.mlpackage \
                     WEIGHTS=runs/coronary/student.pt \
                     IMAGES=data/processed/coronary/val/img MASKS=data/processed/coronary/val/msk
make bench-coreml    MODEL=runs/coronary/student.mlpackage   # latency/fps/size on the Neural Engine
```
The `validate-coreml` step is the **hard gate**: clDice drop after compression must be ≤ 0.03 (INT8/palettization breaks thin vessels even when Dice holds).